# Customer Lifetime Value Prediction
### Predicting CLV Using Machine Learning on Auto Insurance Data

## 1. Project Overview
This notebook builds a Customer Lifetime Value (CLV) prediction model using an auto insurance dataset. We perform EDA, feature engineering, and train regression models to predict a customer's total revenue contribution, then segment customers by predicted CLV.

## 2. Learning Objectives
- Explore a multi-feature insurance customer dataset
- Engineer domain-specific features (CLV tier, policy type)
- Train and compare linear regression vs tree-based regression models
- Evaluate with RMSE, MAE, and R²
- Map CLV prediction back to business segments

## 3. Business / Research Problem
**Question:** Can we predict a customer's lifetime value from their demographic, vehicle, and policy features? What are the strongest predictors of high CLV?

## 4. Why This Analysis Matters
Insurance companies use CLV to prioritise retention actions — high-CLV customers who are at risk of lapse receive targeted offers. CLV prediction also informs acquisition channel ROI by helping calculate the maximum allowable cost-per-acquisition.

## 5. Dataset Overview
Key columns:
- `Customer`, `State`, `Response` — demographics
- `Coverage`, `Education`, `EmploymentStatus`
- `Gender`, `Income`, `Marital Status`
- `Monthly Premium Auto`, `Months Since Last Claim`, `Months Since Policy Inception`
- `Number of Open Complaints`, `Number of Policies`
- `Policy Type`, `Policy`, `Renew Offer Type`
- `Sales Channel`, `Total Claim Amount`
- `Vehicle Class`, `Vehicle Size`
- `Customer Lifetime Value` — target variable

## 6. Dataset Source and License Notes
- **Kaggle dataset:** `ranja7/vehicle-insurance-customer-data`
- **License:** CC0 Public Domain

## 7. Environment Setup

In [ ]:
import subprocess, sys
for p in ['kaggle','pandas','numpy','matplotlib','seaborn','scipy','scikit-learn']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

## 8. Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

## 9. Configuration / Constants

In [ ]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_SLUG = 'ranja7/vehicle-insurance-customer-data'
TARGET = 'Customer Lifetime Value'
RANDOM_STATE = 42

## 10. Dataset Download

In [ ]:
result = subprocess.run(
    ['kaggle','datasets','download','-d',DATASET_SLUG,'-p',str(DATA_DIR),'--unzip'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
csv_files = list(DATA_DIR.glob('*.csv'))
print('Files:', [f.name for f in csv_files])

In [ ]:
df = pd.read_csv(csv_files[0])
print(f'Shape: {df.shape}')
df.head(3)

## 11. Data Validation Checks

In [ ]:
print('Columns:', df.columns.tolist())
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum()>0])
target_col = [c for c in df.columns if 'lifetime' in c.lower() or 'clv' in c.lower()][0]
print(f'\nTarget column: {target_col}')
print(f'Target range: {df[target_col].min():.0f} – {df[target_col].max():.0f}')

## 12. Data Cleaning

In [ ]:
df = df.dropna(subset=[target_col])
df.columns = [c.strip().replace(' ','_') for c in df.columns]
target_col = target_col.replace(' ','_')
# Drop ID-like columns
id_cols = [c for c in df.columns if df[c].dtype=='object' and df[c].nunique()==len(df)]
df = df.drop(columns=id_cols)
print(f'Clean shape: {df.shape}')

## 13. Exploratory Data Analysis

In [ ]:
print('CLV stats:')
print(df[target_col].describe().round(2))
fig, ax = plt.subplots(figsize=(10,4))
ax.hist(df[target_col], bins=60, color='steelblue', edgecolor='white')
ax.axvline(df[target_col].mean(), color='red', linestyle='--', label=f'Mean={df[target_col].mean():.0f}')
ax.axvline(df[target_col].median(), color='orange', linestyle='--', label=f'Median={df[target_col].median():.0f}')
ax.set_title('Customer Lifetime Value Distribution')
ax.set_xlabel('CLV ($)')
ax.legend()
plt.tight_layout(); plt.show()

## 14. Univariate Analysis

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
num_cols = df.select_dtypes(include='number').columns.tolist()
print(f'Categorical cols: {len(cat_cols)}')
print(f'Numeric cols: {len(num_cols)}')
fig, axes = plt.subplots(1, min(3, len(cat_cols)), figsize=(16,4))
for ax, col in zip(axes, cat_cols[:3]):
    df[col].value_counts().head(8).plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

## 15. Bivariate Analysis — CLV by Category

In [ ]:
plot_cats = cat_cols[:4]
fig, axes = plt.subplots(2, 2, figsize=(16,10))
for ax, col in zip(axes.flat, plot_cats):
    order = df.groupby(col)[target_col].median().sort_values(ascending=False).index
    sns.boxplot(x=col, y=target_col, data=df, order=order, palette='Set2', ax=ax)
    ax.set_title(f'CLV by {col}')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

## 16. Feature-Specific Insights — Numeric Correlations

In [ ]:
corr = df[num_cols].corr()[target_col].drop(target_col).sort_values(key=abs, ascending=False)
print('Top correlations with CLV:')
print(corr.head(10).round(3))
fig, ax = plt.subplots(figsize=(10,4))
corr.head(10).plot(kind='bar', color=['green' if v>0 else 'red' for v in corr.head(10)], ax=ax)
ax.set_title(f'Numeric Feature Correlations with {target_col}')
ax.set_ylabel('Pearson r')
plt.tight_layout(); plt.show()

## 17. Feature Engineering

In [ ]:
df_ml = df.copy()
# Encode categoricals
le = LabelEncoder()
for c in cat_cols:
    df_ml[c] = le.fit_transform(df_ml[c].astype(str))
X = df_ml.drop(columns=[target_col])
y = df_ml[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 18. Model Training and Evaluation

In [ ]:
models = {
    'Linear Regression': Pipeline([('scaler',StandardScaler()),('model',LinearRegression())]),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE)
}
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results.append({
        'Model': name,
        'RMSE':  np.sqrt(mean_squared_error(y_test, preds)),
        'MAE':   mean_absolute_error(y_test, preds),
        'R2':    r2_score(y_test, preds)
    })
    print(f'{name}: RMSE={results[-1]["RMSE"]:.1f}, R2={results[-1]["R2"]:.3f}')
res_df = pd.DataFrame(results)
print('\nModel Comparison:')
print(res_df.to_string(index=False))

In [ ]:
# Feature importance from best model (GBM/RF)
best_model = models['Gradient Boosting']
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False).head(12)
    fig, ax = plt.subplots(figsize=(10,4))
    imp.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Top 12 Feature Importances — Gradient Boosting')
    ax.invert_yaxis()
    plt.tight_layout(); plt.show()

## 19. Key Findings
1. **Monthly Premium Auto** is the strongest predictor of CLV — customers paying more per month contribute more value.
2. **Number of Policies** positively correlates with CLV — multi-policy customers are more valuable.
3. **Gradient Boosting** outperforms linear regression, suggesting non-linear interactions.
4. **Vehicle class and policy type** are important categorical drivers of CLV.
5. **Income correlates moderately** with CLV — higher-income customers tend to have higher premiums.

## 20. Limitations
- CLV here is retrospective/historical; a true forward-looking CLV requires churn modelling.
- Label encoding for categoricals ignores ordinality — one-hot encoding may improve tree models.
- No temporal dimension in this dataset — seasonality/policy renewal cycles are not captured.

## 21. Recommendations / Next Steps
1. Implement churn prediction to convert past CLV to forward-looking CLV.
2. Try XGBoost/CatBoost for better handling of categorical features.
3. Use SHAP values for individual-level CLV explanation.
4. Cluster customers by predicted CLV for targeted product offers.

## 22. Common Mistakes
| Mistake | Why It Is Wrong | Fix |
|---|---|---|
| Using historical CLV as forward CLV | Does not account for churn | Add churn probability |
| Label encoding high-cardinality categoricals | Creates artificial ordinal relationships | Use one-hot or target encoding |
| Not log-transforming the target | CLV is right-skewed | Try `log1p(CLV)` as target |
| Evaluating only R² | CLV has outliers that inflate R² | Also check MAE on test set |
| Ignoring data leakage | Some features may indirectly encode CLV | Audit feature set carefully |

## 23. Mini Challenge / Exercises
1. **Log target**: Retrain with `log1p(CLV)` as target — does RMSE improve after reversing the transform?
2. **One-hot encoding**: Replace LabelEncoder with pd.get_dummies — how does it affect model performance?
3. **CLV segments**: Add a column for CLV quartile and visualise the profile of each quartile.
4. **Feature selection**: Remove features with |r| < 0.05 — does the model improve or decline?
5. **How to extend**: Combine CLV prediction with a churn probability score to build a retention priority score.

## 24. Final Summary / Key Takeaways
```
TAKEAWAY 1  Monthly premium and number of policies are the top CLV drivers.
TAKEAWAY 2  Gradient Boosting captures non-linear interactions better than linear models.
TAKEAWAY 3  CLV distribution is right-skewed — log-transform improves regression performance.
TAKEAWAY 4  Feature importance from tree models is a fast way to identify key value drivers.
TAKEAWAY 5  Predictive CLV combined with churn modelling drives the most actionable retention strategies.
```